# Credit Card Fraud Analytics — Predictive Analysis

## Phase 6: Predictive Analysis

This phase introduces supervised fraud classification.

Objectives:
- Create a leakage-safe train/test split
- Build a Logistic Regression baseline
- Compare it with a Random Forest model
- Evaluate performance using fraud-appropriate metrics
- Analyze confusion matrices and threshold behavior
- Compare model ranking quality with PR-AUC and ROC-AUC
- Save model evaluation outputs for reporting and dashboard use

Important:
- The `Class` label is the prediction target.
- The train/test split is performed before fitting preprocessing steps.
- Accuracy is not used as the primary metric because fraud is highly imbalanced.
- Precision, Recall, F1, PR-AUC, ROC-AUC, and the confusion matrix are emphasized.
- V1–V28 are anonymized PCA components and have no direct business meaning.
- The models are analytical baselines, not production fraud-detection systems.


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


## 2. Load Clean Dataset


In [ ]:
DATA_PATH = "../data/creditcard_clean.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Fraud transactions: {(df['Class'] == 1).sum():,}")
print(f"Fraud rate: {df['Class'].mean() * 100:.4f}%")


## 3. Define Target and Features


In [ ]:
target = "Class"

feature_columns = [
    "Time",
    *[f"V{i}" for i in range(1, 29)],
    "Amount"
]

X = df[feature_columns].copy()
y = df[target].copy()

print(f"Feature count: {X.shape[1]}")
print(f"Target distribution:")
display(y.value_counts().rename(index={0: "Normal", 1: "Fraud"}))


## 4. Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training fraud rate: {y_train.mean() * 100:.4f}%")
print(f"Test fraud rate: {y_test.mean() * 100:.4f}%")


The split is stratified so that the rare fraud class is represented proportionally in both training and test sets.

All preprocessing is fitted only on the training data through the modeling pipeline.


## 5. Logistic Regression Baseline


In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        solver="liblinear",
        random_state=42
    ))
])

logistic_pipeline.fit(X_train, y_train)

logistic_prob = logistic_pipeline.predict_proba(X_test)[:, 1]
logistic_pred = (logistic_prob >= 0.50).astype(int)

print("Logistic Regression fitted successfully.")


## 6. Random Forest Model


In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

rf_prob = random_forest.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.50).astype(int)

print("Random Forest fitted successfully.")


## 7. Model Evaluation Function


In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    return {
        "Model": model_name,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Predicted Fraud Count": int(y_pred.sum()),
        "Actual Fraud Count": int(y_true.sum())
    }

model_results = pd.DataFrame([
    evaluate_model(
        y_test,
        logistic_pred,
        logistic_prob,
        "Logistic Regression"
    ),
    evaluate_model(
        y_test,
        rf_pred,
        rf_prob,
        "Random Forest"
    )
])

display(model_results)


## 8. Confusion Matrices


In [ ]:
logistic_cm = confusion_matrix(y_test, logistic_pred)
rf_cm = confusion_matrix(y_test, rf_pred)

logistic_cm_df = pd.DataFrame(
    logistic_cm,
    index=["Actual Normal", "Actual Fraud"],
    columns=["Predicted Normal", "Predicted Fraud"]
)

rf_cm_df = pd.DataFrame(
    rf_cm,
    index=["Actual Normal", "Actual Fraud"],
    columns=["Predicted Normal", "Predicted Fraud"]
)

print("Logistic Regression")
display(logistic_cm_df)

print("Random Forest")
display(rf_cm_df)


## 9. Precision-Recall Curves


In [ ]:
logistic_precision, logistic_recall, _ = precision_recall_curve(
    y_test,
    logistic_prob
)

rf_precision, rf_recall, _ = precision_recall_curve(
    y_test,
    rf_prob
)

plt.figure(figsize=(9, 6))
plt.plot(
    logistic_recall,
    logistic_precision,
    label="Logistic Regression"
)
plt.plot(
    rf_recall,
    rf_precision,
    label="Random Forest"
)

plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 10. ROC Curves


In [ ]:
logistic_fpr, logistic_tpr, _ = roc_curve(
    y_test,
    logistic_prob
)

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_prob
)

plt.figure(figsize=(9, 6))
plt.plot(
    logistic_fpr,
    logistic_tpr,
    label=f"Logistic Regression (AUC={roc_auc_score(y_test, logistic_prob):.4f})"
)
plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC={roc_auc_score(y_test, rf_prob):.4f})"
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Threshold Analysis


In [ ]:
def threshold_metrics(y_true, probabilities, thresholds):
    rows = []

    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)

        rows.append({
            "threshold": threshold,
            "precision": precision_score(
                y_true, predictions, zero_division=0
            ),
            "recall": recall_score(
                y_true, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_true, predictions, zero_division=0
            ),
            "predicted_fraud_count": int(predictions.sum())
        })

    return pd.DataFrame(rows)

thresholds = np.arange(0.05, 1.00, 0.05)

logistic_thresholds = threshold_metrics(
    y_test,
    logistic_prob,
    thresholds
)

rf_thresholds = threshold_metrics(
    y_test,
    rf_prob,
    thresholds
)

display(logistic_thresholds)
display(rf_thresholds)


## 12. Threshold Trade-Off — Logistic Regression


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    logistic_thresholds["threshold"],
    logistic_thresholds["precision"],
    marker="o",
    label="Precision"
)
plt.plot(
    logistic_thresholds["threshold"],
    logistic_thresholds["recall"],
    marker="o",
    label="Recall"
)
plt.plot(
    logistic_thresholds["threshold"],
    logistic_thresholds["f1"],
    marker="o",
    label="F1"
)

plt.title("Logistic Regression Threshold Trade-Off")
plt.xlabel("Classification Threshold")
plt.ylabel("Metric Value")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 13. Threshold Trade-Off — Random Forest


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    rf_thresholds["threshold"],
    rf_thresholds["precision"],
    marker="o",
    label="Precision"
)
plt.plot(
    rf_thresholds["threshold"],
    rf_thresholds["recall"],
    marker="o",
    label="Recall"
)
plt.plot(
    rf_thresholds["threshold"],
    rf_thresholds["f1"],
    marker="o",
    label="F1"
)

plt.title("Random Forest Threshold Trade-Off")
plt.xlabel("Classification Threshold")
plt.ylabel("Metric Value")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 14. Best F1 Threshold


In [ ]:
best_logistic_f1 = logistic_thresholds.loc[
    logistic_thresholds["f1"].idxmax()
]

best_rf_f1 = rf_thresholds.loc[
    rf_thresholds["f1"].idxmax()
]

best_thresholds = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Threshold": best_logistic_f1["threshold"],
        "Precision": best_logistic_f1["precision"],
        "Recall": best_logistic_f1["recall"],
        "F1": best_logistic_f1["f1"],
        "Predicted Fraud Count": best_logistic_f1["predicted_fraud_count"]
    },
    {
        "Model": "Random Forest",
        "Threshold": best_rf_f1["threshold"],
        "Precision": best_rf_f1["precision"],
        "Recall": best_rf_f1["recall"],
        "F1": best_rf_f1["f1"],
        "Predicted Fraud Count": best_rf_f1["predicted_fraud_count"]
    }
])

display(best_thresholds)


The threshold that maximizes F1 is an analytical reference point.

In a real fraud operation, the threshold should be selected according to the relative costs of missed fraud, false alerts, investigation capacity, and customer impact.


## 15. Logistic Regression Coefficients


In [ ]:
logistic_model = logistic_pipeline.named_steps["model"]

coefficient_table = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": logistic_model.coef_[0]
})

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(coefficient_table.head(15))


Logistic regression coefficients show how each feature contributes to the model's linear decision boundary.

Because V1–V28 are anonymized PCA components, coefficient direction should not be translated into a business explanation.


## 16. Random Forest Feature Importance


In [ ]:
rf_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": random_forest.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(rf_importance.head(15))


Feature importance indicates which variables contributed most to the Random Forest splits. It is not a causal explanation.


## 17. Random Forest Feature Importance Plot


In [ ]:
top_rf_features = rf_importance.head(15).sort_values("importance")

plt.figure(figsize=(9, 7))
plt.barh(
    top_rf_features["feature"],
    top_rf_features["importance"]
)

plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 18. Model Comparison


In [ ]:
comparison = model_results.copy()

comparison["Recall_minus_Precision"] = (
    comparison["Recall"] - comparison["Precision"]
)

comparison = comparison.sort_values(
    "PR-AUC",
    ascending=False
)

display(comparison)


For this dataset, PR-AUC is particularly important because the positive class is extremely rare. A model with a strong ROC-AUC can still have weak precision-recall behavior at useful operating points.


## 19. Selected Operating Point


In [ ]:
selected_rows = []

for model_name, probabilities in [
    ("Logistic Regression", logistic_prob),
    ("Random Forest", rf_prob)
]:
    thresholds_df = threshold_metrics(
        y_test,
        probabilities,
        np.arange(0.10, 1.00, 0.05)
    )

    best = thresholds_df.loc[thresholds_df["f1"].idxmax()]

    selected_rows.append({
        "Model": model_name,
        "Selected Threshold": best["threshold"],
        "Precision": best["precision"],
        "Recall": best["recall"],
        "F1": best["f1"],
        "Predicted Fraud Count": best["predicted_fraud_count"]
    })

selected_operating_points = pd.DataFrame(selected_rows)

display(selected_operating_points)


## 20. Final Predictive KPIs


In [ ]:
predictive_kpis = pd.DataFrame({
    "KPI": [
        "Logistic Regression PR-AUC",
        "Logistic Regression ROC-AUC",
        "Logistic Regression Precision",
        "Logistic Regression Recall",
        "Logistic Regression F1",
        "Random Forest PR-AUC",
        "Random Forest ROC-AUC",
        "Random Forest Precision",
        "Random Forest Recall",
        "Random Forest F1"
    ],
    "Value": [
        average_precision_score(y_test, logistic_prob),
        roc_auc_score(y_test, logistic_prob),
        precision_score(y_test, logistic_pred, zero_division=0),
        recall_score(y_test, logistic_pred, zero_division=0),
        f1_score(y_test, logistic_pred, zero_division=0),
        average_precision_score(y_test, rf_prob),
        roc_auc_score(y_test, rf_prob),
        precision_score(y_test, rf_pred, zero_division=0),
        recall_score(y_test, rf_pred, zero_division=0),
        f1_score(y_test, rf_pred, zero_division=0)
    ]
})

display(predictive_kpis)


## 21. Save Phase 6 Outputs


In [ ]:
OUTPUT_DIR = "../data"

model_results.to_csv(
    f"{OUTPUT_DIR}/predictive_model_comparison.csv",
    index=False
)

logistic_cm_df.to_csv(
    f"{OUTPUT_DIR}/logistic_confusion_matrix.csv"
)

rf_cm_df.to_csv(
    f"{OUTPUT_DIR}/random_forest_confusion_matrix.csv"
)

logistic_thresholds.to_csv(
    f"{OUTPUT_DIR}/logistic_threshold_analysis.csv",
    index=False
)

rf_thresholds.to_csv(
    f"{OUTPUT_DIR}/random_forest_threshold_analysis.csv",
    index=False
)

best_thresholds.to_csv(
    f"{OUTPUT_DIR}/best_f1_thresholds.csv",
    index=False
)

coefficient_table.to_csv(
    f"{OUTPUT_DIR}/logistic_coefficients.csv",
    index=False
)

rf_importance.to_csv(
    f"{OUTPUT_DIR}/random_forest_feature_importance.csv",
    index=False
)

selected_operating_points.to_csv(
    f"{OUTPUT_DIR}/selected_operating_points.csv",
    index=False
)

predictive_kpis.to_csv(
    f"{OUTPUT_DIR}/predictive_kpis.csv",
    index=False
)

print("Phase 6 output tables saved successfully.")


## 22. Phase 6 Conclusions

After running the notebook, document evidence-based findings for:

1. Which model has the stronger PR-AUC
2. Which model has the stronger ROC-AUC
3. The precision-recall trade-off at the default threshold
4. The best F1 operating point for each model
5. The number of transactions flagged as fraud at each operating point
6. Which features receive the highest model importance
7. Whether the predictive model is more useful for ranking or direct classification
8. The business trade-off between missed fraud and investigation workload

Avoid claiming that the model is production-ready.

### Next Phase

**Phase 7 — SQL Analysis**

The next phase will translate the analytical questions into SQL queries and business KPIs suitable for a Data Analyst portfolio.
